# Multi-panel figures and compound figure (Solution)

Muc tieu:
- Biet khi nao dung multi-panel (small multiples) de so sanh cong bang.
- Biet tao compound figure co 1 thong diep chinh + panel phu tro.
- Biet quan ly layout, title, annotation, legend nhat quan.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data" / "gapminder.csv").exists():
            return p
    raise FileNotFoundError("Cannot locate data/gapminder.csv")

root = resolve_repo_root()
df = pd.read_csv(root / "data" / "gapminder.csv")
d2007 = df[df["year"] == 2007].copy()
d2007.head()

## 1) Multi-panel co cung truc (fair comparison)

Case: so sanh phan bo `lifeExp` theo continent.
Dung cung x/y limits de tranh gay ao giac khac biet do scale.

In [ ]:
continents = sorted(d2007["continent"].unique())
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
axes = axes.flatten()

for ax, c in zip(axes, continents):
    sub = d2007[d2007["continent"] == c]
    sns.histplot(sub["lifeExp"], bins=12, kde=True, ax=ax, color="#4C78A8")
    ax.set_title(c)

# hide extra subplot slot
for ax in axes[len(continents):]:
    ax.axis("off")

fig.suptitle("Multi-panel: life expectancy distribution by continent (2007)", fontsize=14)
fig.supxlabel("lifeExp")
fig.supylabel("count")
plt.tight_layout()
plt.show()

## 2) Multi-panel voi y nghia so sanh ro (trend theo thoi gian)

Case: trend `lifeExp` theo nam, tach panel theo continent.

In [ ]:
trend = df.groupby(["year", "continent"], as_index=False)["lifeExp"].mean()

g = sns.FacetGrid(trend, col="continent", col_wrap=3, height=3.2, sharex=True, sharey=True)
g.map_dataframe(sns.lineplot, x="year", y="lifeExp", color="#F58518")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("Small multiples: mean lifeExp over time", y=1.03)
plt.show()

## 3) Compound figure (mot thong diep + nhieu panel)

Thong diep: "GDP cao thuong di cung life expectancy cao, nhung phan bo theo khu vuc rat khac nhau".

Cau truc figure:
- Panel A (lon): scatter tong quan GDP vs lifeExp.
- Panel B: bar mean lifeExp theo continent.
- Panel C: boxplot lifeExp theo continent.

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 2, width_ratios=[1.6, 1], height_ratios=[1, 1], hspace=0.28, wspace=0.22)

# Panel A
axA = fig.add_subplot(gs[:, 0])
sns.scatterplot(
    data=d2007,
    x="gdpPercap",
    y="lifeExp",
    hue="continent",
    size="pop",
    sizes=(20, 350),
    alpha=0.75,
    ax=axA,
)
axA.set_xscale("log")
axA.set_title("A) GDP per capita vs life expectancy (2007)", loc="left", fontweight="bold")
axA.set_xlabel("gdpPercap (log scale)")
axA.set_ylabel("lifeExp")

# Panel B
axB = fig.add_subplot(gs[0, 1])
agg = d2007.groupby("continent", as_index=False)["lifeExp"].mean().sort_values("lifeExp", ascending=False)
sns.barplot(data=agg, x="continent", y="lifeExp", hue="continent", palette="Set2", legend=False, ax=axB)
axB.set_title("B) Mean lifeExp by continent", loc="left", fontweight="bold")
axB.tick_params(axis="x", rotation=20)

# Panel C
axC = fig.add_subplot(gs[1, 1])
sns.boxplot(data=d2007, x="continent", y="lifeExp", ax=axC)
axC.set_title("C) Distribution of lifeExp by continent", loc="left", fontweight="bold")
axC.tick_params(axis="x", rotation=20)

fig.suptitle("Compound figure: global pattern + grouped summaries", fontsize=15, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

## 4) Design checklist cho compound figure

- Panel chinh dat ben trai/tren trai, panel phu dat ben phai/duoi.
- Tieu de panel co nhan A/B/C va cung phong cach.
- Truc, don vi, scale nhat quan de doc chinh xac.
- Legend khong lap lai khong can thiet.
- Chu thich ngan gon de noi thong diep, khong de nguoi doc tu doan.

## Reflection

1. Khi nao nen tach thanh multi-panel thay vi ve chung 1 chart?
2. Compound figure cua ban co panel nao la "main evidence"? Vi sao?
3. Neu phai dua vao 1 slide cho manager, ban giu lai panel nao va bo panel nao?